# QuanTA (Quantum-Informed Tensor Adaptation) — GPU Tests

이 노트북은 `quanta/v0.18.1` 브랜치의 QuanTA 통합을 **GPU 환경(Google Colab)**에서 테스트합니다.

**런타임 설정**: 런타임 → 런타임 유형 변경 → T4 GPU (또는 A100)

## 0. 환경 설치

In [ ]:
# peft 소스 설치 (quanta/v0.18.1 브랜치)
!pip install -q git+https://github.com/huggingface/peft.git@quanta/v0.18.1
!pip install -q transformers datasets accelerate

In [ ]:
import torch
import torch.nn as nn

assert torch.cuda.is_available(), "GPU를 사용할 수 없습니다. 런타임을 GPU로 변경하세요."
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")

import peft
print(f"PEFT: {peft.__version__}")
from peft import QuantaConfig, get_peft_model
from peft.tuners.quanta.layer import QuantaLinear
print("QuantaConfig, QuantaLinear import OK")

## 1. 공통 헬퍼

In [ ]:
import warnings

class SimpleMLP(nn.Module):
    def __init__(self, in_features=64, out_features=64):
        super().__init__()
        self.fc = nn.Linear(in_features, out_features, bias=False)

    def forward(self, x):
        return self.fc(x)


def make_peft_model(in_features=64, out_features=64, d=2, per_dim_features=None, quanta_dropout=0.0):
    model = SimpleMLP(in_features, out_features)
    config = QuantaConfig(d=d, per_dim_features=per_dim_features, quanta_dropout=quanta_dropout, target_modules=["fc"])
    return get_peft_model(model, config), model


PASS = "\033[92m✓ PASS\033[0m"
FAIL = "\033[91m✗ FAIL\033[0m"

def check(cond, msg):
    status = PASS if cond else FAIL
    print(f"  {status}  {msg}")
    if not cond:
        raise AssertionError(msg)

print("헬퍼 정의 완료")

## 2. GPU fp16 forward — NaN 없음 확인

로컬에서 CUDA 없어 skip됐던 테스트 (`TestQuantaFP16::test_fp16_forward`)

In [ ]:
print("[Test] GPU fp16 forward — NaN/Inf 없음")

peft_model, _ = make_peft_model(64, 64, d=2)
peft_model = peft_model.half().cuda()
x = torch.randn(4, 64, dtype=torch.float16, device="cuda")

with torch.no_grad():
    y = peft_model(x)

check(torch.isfinite(y).all(), f"fp16 CUDA forward: NaN/Inf 없음 (shape={tuple(y.shape)})")
check(y.dtype == torch.float16, f"출력 dtype = float16")
check(y.device.type == "cuda", f"출력 device = cuda")

## 3. GPU bf16 forward

In [ ]:
print("[Test] GPU bf16 forward — NaN/Inf 없음")

peft_model, _ = make_peft_model(64, 64, d=2)
peft_model = peft_model.to(torch.bfloat16).cuda()
x = torch.randn(4, 64, dtype=torch.bfloat16, device="cuda")

with torch.no_grad():
    y = peft_model(x)

check(torch.isfinite(y).all(), f"bf16 CUDA forward: NaN/Inf 없음 (shape={tuple(y.shape)})")
check(y.dtype == torch.bfloat16, f"출력 dtype = bfloat16")

## 4. GPU zero-init (fp16/bf16)

In [ ]:
print("[Test] GPU zero-init — fp16")
peft_model, base_model = make_peft_model(64, 64, d=2)

peft_model = peft_model.half().cuda()
base_model = base_model.half().cuda()
x = torch.randn(4, 64, dtype=torch.float16, device="cuda")

with torch.no_grad():
    y_peft = peft_model(x)
    y_base = base_model(x)

check(torch.allclose(y_peft, y_base, atol=1e-2), "fp16 초기 delta = 0 (base 출력과 일치)")

print("[Test] GPU zero-init — bf16")
peft_model2, base_model2 = make_peft_model(64, 64, d=2)
peft_model2 = peft_model2.to(torch.bfloat16).cuda()
base_model2 = base_model2.to(torch.bfloat16).cuda()
x2 = torch.randn(4, 64, dtype=torch.bfloat16, device="cuda")

with torch.no_grad():
    y_peft2 = peft_model2(x2)
    y_base2 = base_model2(x2)

check(torch.allclose(y_peft2, y_base2, atol=1e-1), "bf16 초기 delta = 0 (base 출력과 일치)")

## 5. GPU merge/unmerge 정확도

In [ ]:
print("[Test] GPU merge → unmerge 후 weights 복원")

peft_model, _ = make_peft_model(64, 64, d=2)
peft_model = peft_model.cuda()

base_weight_before = peft_model.model.fc.get_base_layer().weight.data.clone()
peft_model.merge_adapter()
peft_model.unmerge_adapter()
base_weight_after = peft_model.model.fc.get_base_layer().weight.data

check(torch.allclose(base_weight_before, base_weight_after, atol=1e-5), "merge→unmerge 후 weight 복원")

print("[Test] GPU merged forward == unmerged forward")
peft_model2, _ = make_peft_model(64, 64, d=2)
peft_model2 = peft_model2.cuda()

for p in peft_model2.parameters():
    if p.requires_grad:
        nn.init.normal_(p)
        break

x = torch.randn(4, 64, device="cuda")
with torch.no_grad():
    y_unmerged = peft_model2(x)

peft_model2.merge_adapter()
with torch.no_grad():
    y_merged = peft_model2(x)

check(torch.allclose(y_unmerged, y_merged, atol=1e-4), "merged vs unmerged forward 일치")

## 6. GPU 역전파 (gradient flow)

In [ ]:
print("[Test] GPU 역전파 — gradient 흐름 확인")

peft_model, _ = make_peft_model(64, 64, d=2)
peft_model = peft_model.cuda()
peft_model.train()

x = torch.randn(8, 64, device="cuda")
target = torch.randn(8, 64, device="cuda")

optimizer = torch.optim.Adam(peft_model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

losses = []
for step in range(5):
    optimizer.zero_grad()
    out = peft_model(x)
    loss = loss_fn(out, target)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

# 훈련 가능한 파라미터에 gradient 있는지 확인
grads = [p.grad for p in peft_model.parameters() if p.requires_grad and p.grad is not None]
check(len(grads) > 0, f"gradient 있는 파라미터 {len(grads)}개")
check(losses[-1] < losses[0], f"loss 감소: {losses[0]:.4f} → {losses[-1]:.4f}")
print(f"  loss 이력: {[f'{l:.4f}' for l in losses]}")

## 7. 대형 레이어 GPU 성능

In [ ]:
import time

print("[Test] 대형 레이어 GPU 성능 측정")

configs = [
    (512, 512, 2, None),
    (1024, 1024, 2, None),
    (4096, 4096, 2, None),
]

for in_f, out_f, d, pdf in configs:
    peft_model, _ = make_peft_model(in_f, out_f, d=d, per_dim_features=pdf)
    peft_model = peft_model.cuda()
    peft_model.eval()

    x = torch.randn(32, in_f, device="cuda")

    # 워밍업
    with torch.no_grad():
        for _ in range(3):
            _ = peft_model(x)
    torch.cuda.synchronize()

    # 측정
    N = 20
    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(N):
            _ = peft_model(x)
    torch.cuda.synchronize()
    elapsed_ms = (time.perf_counter() - start) / N * 1000

    trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in peft_model.parameters())
    print(f"  ({in_f}→{out_f}, d={d}): {elapsed_ms:.2f} ms/iter | "
          f"trainable {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## 8. 실제 LLM: GPT-2 + QuanTA

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

print("[Test] GPT-2 + QuanTA 적용")

model_id = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
base_model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16)

config = QuantaConfig(
    d=2,
    target_modules=["c_attn", "c_proj"],
)
peft_model = get_peft_model(base_model, config)
peft_model = peft_model.cuda()

peft_model.print_trainable_parameters()

# forward pass
inputs = tokenizer("Hello, QuanTA!", return_tensors="pt").to("cuda")
with torch.no_grad():
    out = peft_model(**inputs)

check(torch.isfinite(out.logits).all(), f"GPT-2 QuanTA forward: NaN 없음 (shape={tuple(out.logits.shape)})")

# backward pass
labels = inputs["input_ids"].clone()
peft_model.train()
out_train = peft_model(**inputs, labels=labels)
out_train.loss.backward()

grads = [p.grad for p in peft_model.parameters() if p.requires_grad and p.grad is not None]
check(len(grads) > 0, f"GPT-2 역전파: gradient {len(grads)}개")

## 9. GPU 저장/로드

In [ ]:
import io

print("[Test] GPU — state_dict 저장/로드")

peft_model, _ = make_peft_model(64, 64, d=2)
peft_model = peft_model.cuda()

layer: QuantaLinear = peft_model.model.fc
key = next(iter(layer.quanta_weights2["default"].keys()))
buf_before = layer.quanta_weights2["default"][key].clone()

sd = peft_model.state_dict()
peft_model2, _ = make_peft_model(64, 64, d=2)
peft_model2 = peft_model2.cuda()
peft_model2.load_state_dict(sd)

layer2: QuantaLinear = peft_model2.model.fc
buf_after = layer2.quanta_weights2["default"][key]
check(torch.allclose(buf_before, buf_after.cpu()), "quanta_weights2 GPU→state_dict→GPU 복원")

print("[Test] GPU — torch.save/load (pickle)")
buf = io.BytesIO()
torch.save(peft_model, buf)
buf.seek(0)
peft_model3 = torch.load(buf, weights_only=False)

x = torch.randn(4, 64, device="cuda")
with torch.no_grad():
    y1 = peft_model(x)
    y2 = peft_model3(x)
check(torch.allclose(y1, y2, atol=1e-5), "pickle 후 GPU forward 일치")

## 10. d=3 GPU 테스트

In [ ]:
print("[Test] d=3, GPU zero-init")

# 4*4*4=64
peft_model, base_model = make_peft_model(64, 64, d=3, per_dim_features=[4, 4, 4])
peft_model = peft_model.cuda()
base_model = base_model.cuda()

x = torch.randn(4, 64, device="cuda")
with torch.no_grad():
    y_peft = peft_model(x)
    y_base = base_model(x)

check(torch.allclose(y_peft, y_base, atol=1e-5), "d=3 GPU: 초기 delta = 0")

# gradient
peft_model.train()
out = peft_model(x)
out.sum().backward()
grads = [p.grad for p in peft_model.parameters() if p.requires_grad and p.grad is not None]
check(len(grads) > 0, f"d=3 GPU: gradient {len(grads)}개")

## 11. 결과 요약

In [ ]:
print("=" * 50)
print("QuanTA GPU 테스트 완료")
print("=" * 50)
print("모든 셀이 오류 없이 실행되었으면 테스트 통과입니다.")